In [1]:
"""
08_random_forest.py
"""

'\n08_random_forest.py\n'

In [2]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

df = pd.read_csv(str(project_root / "data" / "processed" / "model_features.csv"))

In [3]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from utils.print_section import print_section

print(df.shape)

# ============================================================
# Features
# ============================================================

FEATURES = [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
    "size",
]

TARGET = "target"

X = df[FEATURES]
y = df[TARGET]

# ============================================================
# Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# ============================================================
# Model
# ============================================================

pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=500,
            random_state=42,
            class_weight="balanced_subsample",
            n_jobs=-1,
        )
    ),
])

# ============================================================
# Fit
# ============================================================

pipeline.fit(
    X_train,
    y_train,
)

# ============================================================
# Predict
# ============================================================

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

# ============================================================
# Performance
# ============================================================

print_section("Performance")

print(
    f"Accuracy : {accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, y_pred):.4f}"
)

print(
    f"F1-score : {f1_score(y_test, y_pred):.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}"
)

# ============================================================
# Confusion matrix
# ============================================================

print_section("Confusion matrix")

cm = confusion_matrix(
    y_test,
    y_pred
)

print(cm)

tn, fp, fn, tp = cm.ravel()

print()

print(f"True Negatives : {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives : {tp:,}")

# ============================================================
# Feature importance
# ============================================================

print_section("Feature importance")

importances = pd.DataFrame({
    "feature": FEATURES,
    "importance": (
        pipeline
        .named_steps["model"]
        .feature_importances_
    )
})

importances = importances.sort_values(
    by="importance",
    ascending=False
)

print(importances)

# ============================================================
# Highest predicted probabilities
# ============================================================

print_section("Highest predicted probabilities")

prob_df = pd.DataFrame({
    "actual": y_test.values,
    "probability": y_prob
})

print(
    prob_df
    .sort_values(
        by="probability",
        ascending=False
    )
    .head(20)
)


(981818, 96)

Performance
Accuracy : 0.9964
Precision: 0.0321
Recall   : 0.0089
F1-score : 0.0139
ROC-AUC  : 0.7257

Confusion matrix
[[195649    151]
 [   559      5]]

True Negatives : 195,649
False Positives: 151
False Negatives: 559
True Positives : 5

Feature importance
         feature  importance
2       solvency    0.266928
0  profitability    0.225124
1      liquidity    0.170424
5           size    0.152214
3      structure    0.100430
4        log_age    0.084880

Highest predicted probabilities
        actual  probability
196304       0     0.922438
68659        0     0.922438
148507       0     0.922438
66631        0     0.922438
37200        0     0.922438
27252        0     0.845229
183491       0     0.845229
134938       0     0.845229
77411        0     0.845229
50896        0     0.845229
52118        0     0.845229
139096       0     0.845229
23899        0     0.792158
14967        0     0.792158
101551       0     0.792158
72311        0     0.792158
54093       